In [2]:
%matplotlib inline

In [59]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_iris
from IPython.display import display

In [48]:
import logging

logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger(__name__)

In [92]:
def load_data():
    """
    Modify the iris dataset for binary classification (predicting 'setosa' only),
    and select only the sepal dimensions (first two features).
    """
    iris = load_iris()
    X = iris['data']
    y = iris['target']

    # Convert to binary classification: predict only 'setosa' (target 0)
    y = (y == 0).astype(int)
    y = y.reshape(-1, 1)

    # Select only sepal dimensions (first two features)
    X = X[:, :2]

    print(f"Loaded data.")

    df = pd.DataFrame(X, columns=iris['feature_names'][:2])
    display('X', X.shape, df.head(2))
    
    df = pd.DataFrame(y, columns=iris['target_names'][:1])
    display('y', y.shape, df.head(2))   
    
    return X, y


X, y = load_data()

Loaded data.


'X'

(150, 2)

,sepal length (cm),sepal width (cm)
0,5.1,3.5
1,4.9,3.0


'y'

(150, 1)

,setosa
0,1
1,1


<br><br><br><br>

# Maths in Neural Networks

There are many resources that explain the math behind neural networks. However, reading multipe ones i was still not confident i understand all details.

The main objective of this project is to explain the topics and ideas that were hard for me to grasp, the gotchas. This is not a full math overview or code implementation.













In [ ]:
class Neuron():
    def __init__(self, w, threshold):
        self.w = w
        self.threshold = threshold

    def activate(self, x):
        w_sum = self.w * x
        if w_sum >= self.threshold:
            return 1
        return 0
    


class Network():
    def __init__(self, neurons, threshold):
        self.neurons = neurons
        self.threshold = threshold

    def feed_inputs(self, inputs):
        activations = [n.activate(i) for n, i in zip(self.neurons, inputs)]
        total = sum(activations)
        return 1 if total >= self.threshold else 0


print("=== AND Gate ===")
network = Network(
    neurons=[Neuron(1, 1), Neuron(1, 1)],
    threshold=2
)
for input1, input2 in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    out = network.feed_inputs([input1, input2])
    print(f"  ({input1}, {input2}) → {out}\n")


## 

## Vectorize a function

Function can take vectors as inputs.

## Vectors in ML

In ML vectors are usually represented as matrixes e.g. (n,) -> (n,1).

In [4]:
W = np.arange(1, 10).reshape((3,3))
x = np.array([2, -1, 4])


W, x, W @ x

(array([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]]),
 array([ 2, -1,  4]),
 array([12, 27, 42]),
 array([12, 27, 42]))

In [7]:
W = np.arange(1, 10).reshape((3,3))
x = np.array([2, -1, 4]).reshape(-1, 1)


W, x, W @ x 

(array([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]]),
 array([[ 2],
        [-1],
        [ 4]]),
 array([[12],
        [27],
        [42]]))

In [9]:
W @ x# , x @ W.T

array([[12],
       [27],
       [42]])

## Calculate neuron activation

I get confused when i try to read a calculation of neuron’s activation in terms of the activations in the previous layer.


Géron [1] transition of activations from one layer to the next is:

$$h_{W,b}(X) = \phi(XW + b)$$

| Item | Description | Shape |
|------|-------------|-------|
| $h_{W,b}$ | Output matrix (all instances) | $(m, p)$ |
| $\phi$ | Activation function (applied elementwise)  | — |
| $X$ | Input matrix (all instances) | $(m, n)$ |
| $W$ | Weight matrix | $(n, p)$ |
| $b$ | Bias vector (broadcast across all rows) | $(1, p)$ |
| $XW$ | Dot product (weighted sum) | $(m, p)$ |
| $XW + b$ | Pre-activation / logits | $(m, p)$ |
| $m$ | number of instances | $(m,)$ |
| $n$ | number of neurons in input layer | $(n,)$ |
| $p$ | number of neurons in current layer | $(p,)$ |


```
h_{W,b}  hypothesis                                 (m, p)
\phi        Activation function 
X        Input matrix                               (m, n)
W        Weight matrix                              (n, p)
b        Bias vector (broadcast across all rows)    (1, p)
XW       Dot product (weighted sum)                 (m, p)
XW + b   Pre-activation / logits                    (m, p)
m        number of instances                        (m,)
n        number of neurons in input layer           (n,)
p        number of neurons in current layer         (p,)
```

Note that:
- $X$ contains all instances, not just a single one.
- The number of instances $m$ is also defined.
- The activation function is not specified - can be sigmoid, ReLU...
- In the weighted sum $XW$, $X$ is first and $W$ is second.

In the concrete example of the MNIST dataset:

$$h_{W,b}(X) = \phi(XW + b)$$

| Item | Description | Shape |
|------|-------------|-------|
| $h_{W,b}(X)$ | Output matrix (all instances) | $(60000, 16)$ |
| $\phi$ | Activation function | — |
| $X$ | Input matrix (all instances) | $(60000, 784)$ |
| $W$ | Weight matrix | $(784, 16)$ |
| $b$ | Bias vector (broadcast across all rows) | $(1, 16)$ |
| $XW$ | Dot product (linear combination) | $(60000, 16)$ |
| $XW + b$ | Pre-activation / logits | $(60000, 16)$ |
| $m$ | number of instances, 60000 for MNIST train set | $(60000,)$ |
| $n$ | number of neurons in input layer, 784 for MNIST (28x28 pixels) | $(784,)$ |
| $p$ | number of neurons in first hidden layer | $(16,)$ |

Note that:
- There are 60K rows in the output, one for each input instance. Each is a vector with 16 values = the activations of the neurons.

$$h_{W,b}(X) = \phi(XW + b)$$

| Item | Description | Shape | Domain |
|------|-------------|-------|--------|
| $m$ | number of instances | $(m,)$ | $\mathbb{Z}^+$ |
| $n$ | number of neurons in input layer | $(n,)$ | $\mathbb{Z}^+$ |
| $p$ | number of neurons in current layer | $(p,)$ | $\mathbb{Z}^+$ |
| $X$ | Input matrix (all instances) | $(m, n)$ | $\mathbb{R}$ |
| $W$ | Weight matrix | $(n, p)$ | $\mathbb{R}$ |
| $XW$ | Dot product (linear combination) | $(m, p)$ | $\mathbb{R}$ |
| $b$ | Bias vector (broadcast across all rows) | $(1, p)$ | $\mathbb{R}$ |
| $XW + b$ | Pre-activation / logits | $(m, p)$ | $\mathbb{R}$ |
| $\phi$ | Activation function (applied element-wise) | — | - |
| $h_{W,b}(X)$ | Output matrix (all instances) | $(m, p)$ | depends on $\phi$ |


$$\sigma\left(W \cdot a^{(0)} + b\right)$$

| Item | Description | Shape | Domain |
|------|-------------|-------|--------|
| $n$ | number of neurons in input layer | $(n,)$ | $\mathbb{Z}^+$ |
| $k$ | number of neurons in next layer | $(k,)$ | $\mathbb{Z}^+$ |
| $a^{(0)}$ | Activation vector of input layer (single instance) | $(n, 1)$ — column vector | $\mathbb{R}$ |
| $W$ | Weight matrix | $(k, n)$ | $\mathbb{R}$ |
| $b$ | Bias vector | $(k, 1)$ | $\mathbb{R}$ |
| $W \cdot a^{(0)} + b$ | Pre-activation / logits | $(k, 1)$ | $\mathbb{R}$ |
| $\sigma$ | Sigmoid (or any activation), applied element-wise | — | - |
| $\sigma\left(W \cdot a^{(0)} + b\right)$ | Activation vector of **next layer** | $(k, 1)$ | $(0, 1)$ |

$$\sigma\left(W \cdot a^{(0)} + b\right)$$

| Item | Description | Shape |
|------|-------------|-------|
| $n$ | number of neurons in input layer | $(n,)$ |
| $k$ | number of neurons in next layer | $(k,)$ |
| $a^{(0)}$ | Activation vector of input layer (single instance) | $(n, 1)$ — column vector |
| $W$ | Weight matrix | $(k, n)$ |
| $b$ | Bias vector | $(k, 1)$ |
| $W \cdot a^{(0)} + b$ | Pre-activation / logits | $(k, 1)$ |
| $\sigma$ | Sigmoid (or any activation), applied element-wise | — |
| $\sigma\left(W \cdot a^{(0)} + b\right)$ | Activation vector of **next layer** | $(k, 1)$ |

Transition of activations from one layer to the next, ALL instances (intput example):

$$h_{W,b}(X) = \phi(XW + b)$$

| Item | Description | Shape |
|------|-------------|-------|
| $m$ | number of instances, 60000 for MNIST train set | $(60000,)$ |
| $n$ | number of neurons in input layer, 784 for MNIST (28x28 pixels) | $(784,)$ |
| $p$ | number of neurons in first hidden layer | $(16,)$ |
| $X$ | Input matrix (all instances) | $(60000, 784)$ |
| $W$ | Weight matrix | $(784, 16)$ |
| $XW$ | Dot product (linear combination) | $(60000, 16)$ |
| $b$ | Bias vector (broadcast across all rows) | $(1, 16)$ |
| $XW + b$ | Pre-activation / logits | $(60000, 16)$ |
| $\phi$ | Activation function (applied element-wise) | — |
| $h_{W,b}(X)$ | Output matrix (all instances) | $(60000, 16)$ |

Transition of activations from one layer to the next, a SINGLE instance (input example):

$$ a^{(1)} = \sigma\left(W \cdot a^{(0)} + b\right)$$

| Item | Description | Shape |
|------|-------------|-------|
| $n$ | number of neurons in input layer, 784 for MNIST (28x28 pixels) | $(784,)$ |
| $k$ | number of neurons in first hidden layer | $(16,)$ |
| $a^{(0)}$ | Activation vector of input layer (single image, 784 pixels) | $(784, 1)$ — column vector |
| $W$ | Weight matrix | $(16, 784)$ |
| $W \cdot a^{(0)}$ | Linear combination (matrix-vector multiply) | $(16, 1)$ |
| $b$ | Bias vector | $(16, 1)$ |
| $W \cdot a^{(0)} + b$ | Pre-activation / logits | $(16, 1)$ |
| $\sigma$ | Sigmoid (or any activation), applied element-wise | — |
| $\sigma\left(W \cdot a^{(0)} + b\right)$ | Activation vector of first hidden layer | $(16, 1)$ |

## Choose learning rate

the learning rate, which we had to choose 
carefully to balance the speed of learning against the risk of overshooting the global minimum of the 
loss function.


$w_{in \to out}$ or $w_{layer1-layer2}$
$$
z^{[l]} = W^{[l]} a^{[l-1]} + b^{[l]}
$$

## Four Methods for Computing Derivatives

Computing dy/dx₁ for y = ln(x₁) + x₁x₂ − sin(x₂)

### The True Answer (Analytical)
$$\frac{\partial y}{\partial x_1} = \frac{1}{x_1} + x_2$$

At **x₁ = 2, x₂ = 5** → `1/2 + 5 = 5.5`

---

### (a) Manual Differentiation

$$\frac{\partial}{\partial x_1}[\ln(x_1)] = \frac{1}{x_1}$$

$$\frac{\partial}{\partial x_1}[x_1 x_2] = x_2$$

$$\frac{\partial}{\partial x_1}[\sin(x_2)] = 0 \quad \text{(no x₁)}$$

$$\boxed{\frac{\partial y}{\partial x_1} = \frac{1}{x_1} + x_2}$$

```python
import math

def manual_derivative(x1, x2):
    return 1/x1 + x2

x1, x2 = 2.0, 5.0
print(f"Manual:   {manual_derivative(x1, x2)}")   # 5.5
```

---

### (b) Numerical Differentiation

$$\frac{\partial y}{\partial x_1} \approx \frac{f(x_1 + h,\ x_2) - f(x_1,\ x_2)}{h}$$

```python
import math

def f(x1, x2):
    return math.log(x1) + x1 * x2 - math.sin(x2)

def numerical_derivative(x1, x2, h=1e-5):
    return (f(x1 + h, x2) - f(x1, x2)) / h

x1, x2 = 2.0, 5.0
print(f"Numerical: {numerical_derivative(x1, x2)}")  # ≈ 5.49999...
```

> ⚠️ Not exact — error depends on `h`. Too small → floating point issues. Too large → bad approximation.

---

### (c) Symbolic Differentiation

```python
import sympy as sp

x1, x2 = sp.symbols('x1 x2')

f = sp.ln(x1) + x1 * x2 - sp.sin(x2)

df_dx1 = sp.diff(f, x1)
print(f"Symbolic (formula): {df_dx1}")          # 1/x1 + x2

# Evaluate at x1=2, x2=5
value = df_dx1.subs([(x1, 2), (x2, 5)])
print(f"Symbolic (value):   {value}")            # 11/2 = 5.5
```

Output:
```
Symbolic (formula): x2 + 1/x1
Symbolic (value):   11/2
```

---

### (d) Automatic Differentiation

```python
import torch

x1 = torch.tensor(2.0, requires_grad=True)
x2 = torch.tensor(5.0, requires_grad=True)

y = torch.log(x1) + x1 * x2 - torch.sin(x2)

y.backward()   # reverse mode — computes ALL partials in one pass

print(f"AutoDiff dy/dx1: {x1.grad.item()}")   # 5.5
print(f"AutoDiff dy/dx2: {x2.grad.item()}")   # x1 - cos(x2) = 2 - cos(5)
```

---

### Results Comparison at x₁=2, x₂=5

| Method | dy/dx₁ | Notes |
|---|---|---|
| Manual | `5.5` | Exact, done by hand |
| Numerical | `5.49999...` | Approximate |
| Symbolic | `11/2 = 5.5` | Exact, algebraic |
| **AutoDiff** | `5.5` | **Exact, computed automatically** |

---

### Why AutoDiff Wins in Practice

For a function with **millions of parameters** (neural network):
- Manual → impossible
- Numerical → need one `f()` call **per parameter** → too slow  
- Symbolic → expression size **explodes**
- **AutoDiff reverse mode** → computes **all gradients in ONE backward pass** ✅

## References

[1] Géron, A. (2019). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (2nd ed.). O'Reilly Media.
